# 01 — Download MPI-LEMON from OpenNeuro

**Dataset:** `ds000221` on [openneuro.org](https://openneuro.org/datasets/ds000221)  
**Strategy:** Process one subject at a time — download raw BOLD → preprocess → save features → delete raw.  
This keeps disk usage under ~5 GB at any point, which fits Colab's free tier.

**What we download per subject:**
- `sub-XXXXXX/func/sub-XXXXXX_task-rest_bold.nii.gz` — 4D resting-state BOLD (~300 MB each)
- `participants.tsv` — phenotypic targets (STAI, NEO-FFI, TICS, etc.)

**Output saved to Google Drive:** `lemon/phenotypic.csv` + `lemon/raw/sub-XXXXXX_bold.nii.gz` (temporary)

In [ ]:
# Install dependencies
!pip install -q awscli nibabel nilearn pandas numpy tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/lemon'
RAW_DIR  = f'{BASE_DIR}/raw'
os.makedirs(RAW_DIR, exist_ok=True)
print('Drive mounted. Working directory:', BASE_DIR)

In [ ]:
import subprocess, pandas as pd

# ── Download participants.tsv ─────────────────────────────────────────────────
# LEMON is hosted on OpenNeuro's public S3 bucket — no credentials needed.
BUCKET = 's3://openneuro.org/ds000221'

!aws s3 cp {BUCKET}/participants.tsv {BASE_DIR}/participants.tsv --no-sign-request

participants = pd.read_csv(f'{BASE_DIR}/participants.tsv', sep='\t')
print(f'Total participants: {len(participants)}')
print(participants.head())
print('\nColumns:', participants.columns.tolist())

In [ ]:
# ── List available subjects on S3 ─────────────────────────────────────────────
result = subprocess.run(
    ['aws', 's3', 'ls', f'{BUCKET}/', '--no-sign-request'],
    capture_output=True, text=True
)
lines = result.stdout.strip().split('\n')
subjects = sorted([
    l.strip().rstrip('/').split()[-1]
    for l in lines if 'sub-' in l
])
print(f'Subjects found on S3: {len(subjects)}')
print('First 5:', subjects[:5])

In [ ]:
import numpy as np

# ── Identify phenotypic columns ───────────────────────────────────────────────
# LEMON participants.tsv contains scores from multiple questionnaires.
# We target three wellbeing dimensions that map to the app's output:
#   • Trait anxiety  → STAI_T (or STAI_Trait)
#   • Chronic stress → TICS_SSCS (sum score chronic stress)
#   • Neuroticism    → NEO_N
#
# Column names vary slightly across dataset versions — find them dynamically.

TARGET_PATTERNS = {
    'trait_anxiety':   ['STAI_T', 'STAI_Trait', 'stai_t', 'trait_anxiety'],
    'chronic_stress':  ['TICS_SSCS', 'TICS', 'tics_sscs', 'chronic_stress'],
    'neuroticism':     ['NEO_N', 'neo_n', 'neuroticism', 'NEO_FFI_N'],
}

found = {}
for name, patterns in TARGET_PATTERNS.items():
    for p in patterns:
        matches = [c for c in participants.columns if p.lower() in c.lower()]
        if matches:
            found[name] = matches[0]
            print(f'  {name:20s} → "{matches[0]}", sample values: {participants[matches[0]].dropna().values[:5]}')
            break
    if name not in found:
        print(f'  {name:20s} → NOT FOUND. Available columns: {participants.columns.tolist()}')

# Save the column mapping for use in later notebooks
import json
with open(f'{BASE_DIR}/target_columns.json', 'w') as f:
    json.dump(found, f, indent=2)
print('\nColumn map saved to target_columns.json')

In [ ]:
# ── Filter to subjects that have all three targets ────────────────────────────
target_cols = list(found.values())
valid = participants.dropna(subset=target_cols).copy()
valid_ids = set(valid['participant_id'].str.replace('sub-', '', regex=False).tolist())

# Cross-reference with subjects that actually have func data on S3
subjects_with_func = []
for s in subjects:
    sub_id = s.replace('sub-', '')
    if sub_id in valid_ids:
        subjects_with_func.append(s)

print(f'Subjects with phenotypic data + func data: {len(subjects_with_func)}')

# Save the list for use in later notebooks
with open(f'{BASE_DIR}/subject_list.json', 'w') as f:
    json.dump(subjects_with_func, f, indent=2)

# Also save the filtered phenotypic CSV
valid.to_csv(f'{BASE_DIR}/phenotypic.csv', index=False)
print('subject_list.json and phenotypic.csv saved to Drive.')

In [ ]:
# ── Test: download a single subject to verify the pipeline ───────────────────
# Run this cell before running the full loop in notebook 02.

test_sub = subjects_with_func[0]
bold_key = f'{BUCKET}/{test_sub}/func/{test_sub}_task-rest_bold.nii.gz'
out_path = f'{RAW_DIR}/{test_sub}_bold.nii.gz'

print(f'Downloading {test_sub}...')
!aws s3 cp {bold_key} {out_path} --no-sign-request

import nibabel as nib
img = nib.load(out_path)
print(f'Shape: {img.shape}  — should be (x, y, z, timepoints)')
print(f'TR: {img.header.get_zooms()[3]:.2f}s')
print('Test download successful ✓')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('='*55)
print('DOWNLOAD NOTEBOOK COMPLETE')
print('='*55)
print(f'  Subjects ready for preprocessing: {len(subjects_with_func)}')
print(f'  Phenotypic targets: {list(found.keys())}')
print(f'  Files saved to: {BASE_DIR}')
print()
print('Next step: open 02_preprocess.ipynb')
print('It will download each subject, preprocess, extract features,')
print('and delete the raw file — keeping Drive usage low.')